In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "XGenerationLab/XiYanSQL-QwenCoder-3B-2502"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizer Loaded")

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Tokenizer Loaded


In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")
print("Model Loaded")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Model Loaded


In [ ]:
schema = {
    "customers": [
        "customer_id",
        "customer_name",
        "segment",
        "country",
        "state"
    ],

    "products": [
        "product_key",
        "product_id",
        "product_name",
        "category",
        "sub_category"
    ],

    "sales": [
        "sales_record_id",
        "order_id",
        "customer_id",
        "product_key",
        "order_date",
        "ship_date",
        "ship_mode",
        "market",
        "region",
        "quantity",
        "sales",
        "discount",
        "profit",
        "shipping_cost",
        "order_priority",
        "year"
    ]
}

In [ ]:
relationships = """
Relationships:
- customers.customer_id = sales.customer_id
- products.product_key = sales.product_key
"""

In [ ]:
def format_schema(schema):

  text = ""

  for table, columns in schema.items():

    text += f"{table}(\n"

    for column in columns:
        text += f"    {column},\n"

    text = text.rstrip(",\n")
    text += "\n)\n\n"

  return text

In [ ]:
schema_text = format_schema(schema)

print(schema_text)

customers(
    customer_id,
    customer_name,
    segment,
    country,
    state
)

products(
    product_key,
    product_id,
    product_name,
    category,
    sub_category
)

sales(
    sales_record_id,
    order_id,
    customer_id,
    product_key,
    order_date,
    ship_date,
    ship_mode,
    market,
    region,
    quantity,
    sales,
    discount,
    profit,
    shipping_cost,
    order_priority,
    year
)




In [ ]:
def build_prompt(question):

    prompt = f"""
You are a Text-to-SQL system.

Generate a valid MySQL SQL query to answer the user's question.

Database schema:

{schema_text}

{relationships}

Rules:
1. Generate MySQL SQL only.
2. Use only the tables and columns provided in the schema.
3. Use the provided relationships when JOINs are required.
4. Do not generate INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE, CREATE, or other data-modifying statements.
5. The sales column already represents the sales amount for a sales record. Do not multiply sales by quantity.
6. When asked for the number of orders, count DISTINCT order_id.
7. Use aggregation functions such as SUM(), COUNT(), AVG(), MIN(), and MAX() when appropriate.
8. Return one SQL query that directly answers the question.

Example:

Question:
Which product category generated the highest total profit?

SQL:
SELECT
    p.category,
    SUM(s.profit) AS total_profit
FROM sales s
JOIN products p
    ON s.product_key = p.product_key
GROUP BY p.category
ORDER BY total_profit DESC
LIMIT 1;

Now answer this question.

Question:
{question}

SQL:
"""

    return prompt

In [ ]:
question = "Which region generated the most profit?"

prompt = build_prompt(question)

print(prompt)


You are a Text-to-SQL system.

Generate a valid MySQL SQL query to answer the user's question.

Database schema:

customers(
    customer_id,
    customer_name,
    segment,
    country,
    state
)

products(
    product_key,
    product_id,
    product_name,
    category,
    sub_category
)

sales(
    sales_record_id,
    order_id,
    customer_id,
    product_key,
    order_date,
    ship_date,
    ship_mode,
    market,
    region,
    quantity,
    sales,
    discount,
    profit,
    shipping_cost,
    order_priority,
    year
)




Relationships:
- customers.customer_id = sales.customer_id
- products.product_key = sales.product_key


Rules:
- Generate MySQL SQL only.
- Use only the tables and columns provided.
- Use the relationships provided when JOINs are required.
- Do not generate INSERT, UPDATE, DELETE, DROP, ALTER, or other data-modifying statements.
- Answer the user's question directly.

Example:

Question:
Which product category generated the highest total profit?

SQ

In [ ]:
def generate_sql(question):

    prompt = build_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # Move inputs to the same device as the model
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    # Only take the newly generated tokens
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    sql = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return sql.strip()

In [ ]:
question = "Which region generated the most profit?"

sql = generate_sql(question)

print(sql)

SELECT region, SUM(profit) AS total_profit FROM sales GROUP BY region ORDER BY total_profit DESC LIMIT 1; ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ``` ```


In [ ]:
import re

def clean_sql(sql):
    # Remove markdown code fences
    sql = re.sub(r"```sql", "", sql, flags=re.IGNORECASE)
    sql = sql.replace("```", "")

    # Remove excessive whitespace
    sql = re.sub(r"\s+", " ", sql).strip()

    # Keep only the first SQL statement
    if ";" in sql:
        sql = sql.split(";")[0] + ";"

    return sql

In [ ]:
cleaned_sql = clean_sql(sql)

print("Cleaned SQL:")
print(cleaned_sql)

Cleaned SQL:
SELECT region, SUM(profit) AS total_profit FROM sales GROUP BY region ORDER BY total_profit DESC LIMIT 1;


In [ ]:
import re

ALLOWED_TABLES = {
    "customers",
    "products",
    "sales"
}

FORBIDDEN_KEYWORDS = {
    "INSERT",
    "UPDATE",
    "DELETE",
    "DROP",
    "ALTER",
    "TRUNCATE",
    "CREATE",
    "REPLACE",
    "GRANT",
    "REVOKE"
}


def validate_sql(sql):
    sql_upper = sql.upper()

    # 1. Query must start with SELECT
    if not sql_upper.strip().startswith("SELECT"):
        return False, "Only SELECT queries are allowed."

    # 2. Block dangerous SQL commands
    for keyword in FORBIDDEN_KEYWORDS:
        if re.search(r"\b" + keyword + r"\b", sql_upper):
            return False, f"Forbidden SQL keyword detected: {keyword}"

    # 3. Extract tables used after FROM or JOIN
    tables = re.findall(
        r"\b(?:FROM|JOIN)\s+([a-zA-Z_][a-zA-Z0-9_]*)",
        sql,
        flags=re.IGNORECASE
    )

    # 4. Check that every table exists in our schema
    for table in tables:
        if table.lower() not in ALLOWED_TABLES:
            return False, f"Unknown table detected: {table}"

    return True, "SQL query is valid."

In [ ]:
is_valid, message = validate_sql(cleaned_sql)

print("Valid:", is_valid)
print("Message:", message)

Valid: True
Message: SQL query is valid.


In [ ]:
test_queries = [
    cleaned_sql,

    'DROP TABLE sales;',

    'DELETE FROM sales WHERE sales_record_id = 1;',

    'INSERT INTO sales;',

    'SELECT * FROM employees;'
]

for query in test_queries:
    valid, message = validate_sql(query)
    print("\nQuery:", query)
    print("Valid:", valid)
    print("Message:", message)


Query: SELECT region, SUM(profit) AS total_profit FROM sales GROUP BY region ORDER BY total_profit DESC LIMIT 1;
Valid: True
Message: SQL query is valid.

Query: DROP TABLE sales;
Valid: False
Message: Only SELECT queries are allowed.

Query: DELETE FROM sales WHERE sales_record_id = 1;
Valid: False
Message: Only SELECT queries are allowed.

Query: INSERT INTO sales;
Valid: False
Message: Only SELECT queries are allowed.

Query: SELECT * FROM employees;
Valid: False
Message: Unknown table detected: employees


In [ ]:
test_questions = [
    "Which region generated the most profit?",
    "What is the total sales?",
    "Which product category has the highest sales?",
    "How many orders were placed?",
    "Which customer generated the highest profit?"
]

for question in test_questions:
    print("\n" + "=" * 80)
    print("Question:", question)

    raw_sql = generate_sql(question)
    cleaned = clean_sql(raw_sql)
    valid, message = validate_sql(cleaned)

    print("SQL:", cleaned)
    print("Valid:", valid)
    print("Message:", message)


Question: Which region generated the most profit?
SQL: SELECT region, SUM(profit) AS total_profit FROM sales GROUP BY region ORDER BY total_profit DESC LIMIT 1;
Valid: True
Message: SQL query is valid.

Question: What is the total sales?
SQL: SELECT SUM(sales) AS total_sales FROM sales;
Valid: True
Message: SQL query is valid.

Question: Which product category has the highest sales?
SQL: SELECT p.category, SUM(s.quantity * s.sales) AS total_sales FROM sales s JOIN products p ON s.product_key = p.product_key GROUP BY p.category ORDER BY total_sales DESC LIMIT 1;
Valid: True
Message: SQL query is valid.

Question: How many orders were placed?
SQL: SELECT COUNT(order_id) AS total_orders FROM sales;
Valid: True
Message: SQL query is valid.

Question: Which customer generated the highest profit?
SQL: SELECT c.customer_name, SUM(s.profit) AS total_profit FROM sales s JOIN customers c ON s.customer_id = c.customer_id GROUP BY c.customer_name ORDER BY total_profit DESC LIMIT 1;
Valid: True
Me

In [ ]:
questions = [
    "Which product category has the highest sales?",
    "How many orders were placed?"
]

for question in questions:
    print("\n" + "=" * 80)
    print("Question:", question)

    raw_sql = generate_sql(question)
    cleaned = clean_sql(raw_sql)
    valid, message = validate_sql(cleaned)

    print("SQL:", cleaned)
    print("Valid:", valid)
    print("Message:", message)


Question: Which product category has the highest sales?
SQL: SELECT p.category, SUM(s.sales) AS total_sales FROM sales s JOIN products p ON s.product_key = p.product_key GROUP BY p.category ORDER BY total_sales DESC LIMIT 1;
Valid: True
Message: SQL query is valid.

Question: How many orders were placed?
SQL: SELECT COUNT(DISTINCT order_id) AS total_orders FROM sales;
Valid: True
Message: SQL query is valid.
